# Behavior Modeling API: LimSim Implementation

## Introduction

Tactics2D provides unified reimplementations of a collection of representative traffic participant behavior models to support the development, validation, and testing of Autonomous Driving Systems (ADS) with realistic and scalable traffic interactions. LimSim is one of the default behavior models integrated into Tactics2D.

Original paper: [LimSim: A Long-term Interactive Multi-scenario Traffic Simulator](https://arxiv.org/abs/2307.06648)

Original code: [PJLab-ADG/LimSim](https://github.com/PJLab-ADG/LimSim)

LimSim adopts a hierarchical Prediction–Decision-making–Planning (PDP) framework that decomposes vehicle behavior generation into three sequential stages: prediction, decision-making, and planning. To efficiently model multi-agent interactions, LimSim first partitions globally interacting vehicles into interaction groups and then performs joint decision-making within each group using a group-based Monte Carlo Tree Search (MCTS) algorithm. This hierarchical design enables long-term, interactive traffic simulation while maintaining computational efficiency.

The original paper validates LimSim on both the Waymo Open Motion Dataset (WOMD) and CitySim, demonstrating its capability for long-term interactive traffic simulation across multiple urban scenarios. Rather than reproducing only these benchmark settings, Tactics2D reimplements LimSim on top of its unified data representation, map abstraction, and scenario interface. Consequently, the behavior model can be executed on all datasets supported by Tactics2D, making LimSim one of the framework's default foundational behavior models and providing a consistent behavior generation pipeline across heterogeneous traffic datasets.

In this documentation, we will demonstrate how to use the LimSim behavior model in Tactics2D with all built-in modules and datasets.

## Environment Setup

Please install Tactics2D (`pip install 'tactics2[behavior]'`) or add the Tactics2D source directory to your `PYTHONPATH`. See the [Installation Guide](https://tactics2d.readthedocs.io/en/latest/installation/) for more details.

## Dataset Preparation

Tactics2D does not require datasets to be stored in a fixed location. You can place a dataset in any directory and provide its path when loading it. Throughout this tutorial, **WOMD** is used to demonstrate the standard workflow of LimSim, while **HighD, InD** serves as an additional example to show how LimSim can be used with other datasets supported by Tactics2D.

## Use LimSim for Behavior Generation

The pipeline below demonstrates a complete LimSim takeover visualization. Every step that interacts with data (parsing, representing, predicting, and rendering) is handled by Tactics2D's public API. The notebook only provides glue code.

| Module | Key API |
|--------|---------|
| **Dataset parsers** | `parser.parse_trajectory(...)` → `(participants, time_range)` |
| **Map abstraction** | `map_.roadlines`, `map_.lanes`, `map_.areas` |
| **Participant model** | `participant.trajectory.get_state(frame)` |
| **Behavior model** | `model.predict(participants, map_, frame, agent_ids)` → `{agent_id: Trajectory}` |
| **BEVCamera** | `camera.update(frame, participants, ...)` → `geometry_data` |
| **MatplotlibRenderer** | `renderer.update(geometry_data)` |
| **Trajectory gradient** | `renderer.draw_gradient_trace(positions, colormap)` |
| **Color & style** | `participant.color = "purple"` overrides `COLOR_PALETTE` defaults |

In [ ]:
%matplotlib notebook

import warnings

warnings.filterwarnings("ignore")

import logging

logging.basicConfig(level=logging.WARNING)

import matplotlib as mpl
import matplotlib.cm as cm
from matplotlib import pyplot as plt
from matplotlib.animation import FuncAnimation
from shapely.geometry import Point
import pandas as pd
import numpy as np
import seaborn as sns

from tactics2d.dataset_parser import WOMDParser, LevelXParser
from tactics2d.behavior import LimSimBehaviorModel, LimSimConfig
from tactics2d.display.sensor import BEVCamera
from tactics2d.display.renderers import MatplotlibRenderer
from tactics2d.map.parser import OSMParser
from tactics2d.map.element import Map
from tactics2d.map.map_config import HIGHD_MAP_CONFIG, IND_MAP_CONFIG
from tactics2d.participant.element import Vehicle

pygame 2.6.1 (SDL 2.28.4, Python 3.9.25)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [ ]:
mpl.rcParams.update(
    {
        "figure.dpi": 200,
        "font.family": "DejaVu Sans Mono",
        "font.size": 8,
        "animation.html": "html5",
        "animation.embed_limit": 100 * 1024 * 1024,
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,
        "axes.facecolor": "white",
        "figure.facecolor": "white",
    }
)
sns.set_palette("husl")

In [3]:
# LimSim model config
LIMSIM_CFG = LimSimConfig(
    horizon_steps=10, dt=0.1, mcts_iterations=15, interaction_distance=30.0, max_group_size=4
)

# Prediction / display constants
NUM_SECONDS = 20  # how many seconds of playback
MAX_WORKERS = 8  # parallel predict threads
PERCEPTION_RANGE = 200  # BEVCamera perception range (meters)
VIEW_WIDTH = 120  # local view width centered on ego (meters)
VIEW_HEIGHT = 80  # local view height centered on ego (meters)

## Usage

The following three steps demonstrate a reusable pipeline for visualizing LimSim's predicted trajectories. Each step describes the relevant built-in modules provided by Tactics2D and how they are combined in the example code. For simplicity, this demo takes over only **one vehicle** at a time. To control multiple vehicles simultaneously, simply pass a list of agent IDs to the `model.predict()` function.

### Step 1: Select the Takeover Vehicle

`Vehicle` carries a `Trajectory` (a `dict[int, State]` mapping timestamps to positions) and a `color` attribute consumed by the renderer. `select_takeover_vehicle` picks the vehicle with the longest `history_states`; pass `ego_id` to `run_takeover_scenario` to override this. The color is resolved against `COLOR_PALETTE` — `"purple"` → `#8854d0`, the same mechanism that maps vehicle types to default colors.

In [4]:
def select_takeover_vehicle(participants):
    """Return the vehicle with the longest trajectory, or raise if none found."""
    vehicle_ids = [
        pid
        for pid, p in participants.items()
        if isinstance(p, Vehicle) and len(p.trajectory.history_states) > 0
    ]
    if not vehicle_ids:
        raise ValueError("No vehicle participant found.")
    return max(vehicle_ids, key=lambda pid: len(participants[pid].trajectory.history_states))

### Step 2: Predict Trajectories with LimSim

`LimSimBehaviorModel` wraps the PDP framework (prediction → decision → planning) behind a single call shared by all Tactics2D behavior models:

```python
predicted = model.predict(participants, map_, frame=frame, agent_ids=[ego_id])
```

- `participants` — the same dict from `parse_trajectory()`; each `Vehicle`'s history states up to `frame` are the model's input.
- `map_` — the `Map` from `parse_map()`, providing lane, roadline, and area geometry.
- `agent_ids` — which participants to predict for; others are still observed as context.

Returns `{agent_id: Trajectory}` with predicted future states using the same `.get_state(frame)` API as history.

For multi-frame prediction, use the built-in `predict_batch()` method, which dispatches frames across a `ThreadPoolExecutor` when `parallel_workers` > 1:

```python
model = LimSimBehaviorModel(config, parallel_workers=8)
frame_predictions = model.predict_batch(participants, map_, playback_frames, agent_ids=[ego_id])
```

### Step 3: Render with BEVCamera and MatplotlibRenderer

**BEVCamera** handles viewport culling, coordinate transforms, and geometry generation. `camera.update(...)` returns `(geometry_data, road_set, participant_set)` — the renderer consumes `geometry_data` directly.

**MatplotlibRenderer** draws the scene with automatic z-ordering (road areas < lines < buildings < vehicles < point clouds) and resolves colors from `COLOR_PALETTE` and `DEFAULT_COLOR` by element type.

**Trajectory gradient** (`renderer.enable_trajectory_gradient()` + `draw_gradient_trace()`) renders colormapped past/future traces at zorder 5 — near segments get the dark end, far segments the light end.

The `update()` closure below is called by `FuncAnimation` on every frame: clear old traces → collect active participants → get ego pose → `camera.update()` → `renderer.update()` → draw past trace (BuPu, reversed so closest point is first) → draw future prediction (GnBu).

In [5]:
def render_takeover_animation(
    participants, map_, playback_frames, ego_id, mpc_plans=None, resolution=(1200, 800)
):
    for roadline in map_.roadlines.values():
        if roadline.type_ is None:
            roadline.type_ = "roadline"
    camera = BEVCamera(id_=0, map_=map_, perception_range=PERCEPTION_RANGE)
    prev_road, prev_part = set(), set()
    renderer = MatplotlibRenderer(
        xlim=(-VIEW_WIDTH / 2, VIEW_WIDTH / 2),
        ylim=(-VIEW_HEIGHT / 2, VIEW_HEIGHT / 2),
        resolution=resolution,
        auto_scale=False,
    )
    renderer.enable_trajectory_gradient()
    ego_traj = participants[ego_id].trajectory
    plan_frames = sorted(mpc_plans.keys()) if mpc_plans else []

    def update(frame):
        nonlocal prev_road, prev_part
        renderer._remove_trajectory_lines()
        pids = [pid for pid, p in participants.items() if frame in p.trajectory.history_states]
        if ego_traj.has_state(frame):
            ego_pose = ego_traj.get_state(frame)
            cam_pos = Point(ego_pose.x, ego_pose.y)
            renderer.ax.set_xlim(ego_pose.x - VIEW_WIDTH / 2, ego_pose.x + VIEW_WIDTH / 2)
            renderer.ax.set_ylim(ego_pose.y - VIEW_HEIGHT / 2, ego_pose.y + VIEW_HEIGHT / 2)
        else:
            ego_pose = None
            cam_pos = Point(0.0, 0.0)
        gd, prev_road, prev_part = camera.update(
            frame, participants, pids, prev_road, prev_part, cam_pos
        )
        renderer.update(gd)
        if ego_pose is not None:
            past = [f for f in playback_frames if f <= frame and ego_traj.has_state(f)]
            if past:
                pts = [(ego_traj.get_state(f).x, ego_traj.get_state(f).y) for f in past]
                renderer.draw_gradient_trace(list(reversed(pts)), cm.BuPu)
            if plan_frames:
                cand = [f for f in plan_frames if f <= frame]
                if cand:
                    dxdy = mpc_plans[cand[-1]]
                    fpts = [(ego_pose.x, ego_pose.y)]
                    fpts.extend((ego_pose.x + dx, ego_pose.y + dy) for dx, dy in dxdy)
                    renderer.draw_gradient_trace(fpts, cm.GnBu)
        renderer.ax.set_title(
            f"LimSim takeover: {ego_id}  |  frame {frame}  |  active: {len(pids)}", fontsize=7
        )

    return FuncAnimation(renderer.fig, update, frames=playback_frames, interval=100, repeat=True)

In [6]:
def run_takeover_scenario(
    parser,
    file_name=None,
    folder=None,
    map_path=None,
    map_config=None,
    ego_id=None,
    ego_color="light-pink",
    fps=10,
    num_seconds=NUM_SECONDS,
    resolution=(1200, 800),
    **parse_kwargs,
):
    print("Parsing scenario ...")
    participants, time_range = parser.parse_trajectory(
        file=file_name, folder=folder, **parse_kwargs
    )

    map_ = None
    if hasattr(parser, "parse_map"):
        map_ = parser.parse_map(file=file_name, folder=folder, **parse_kwargs)
    if map_ is None and map_path is not None:
        print(f"  loading map from {map_path}")
        map_ = OSMParser(lanelet2=True).parse(file_path=map_path, configs=map_config)
    if map_ is None:
        map_ = Map("empty_map", scenario_type="demo")

    print(f"  participants: {len(participants)},  frames: {time_range}")
    if ego_id is None:
        ego_id = select_takeover_vehicle(participants)
    participants[ego_id].color = ego_color
    print(f"  takeover target: {ego_id}  (color: {ego_color})")

    # ---- prepare takeover window ----
    ego = participants[ego_id]
    frames = sorted(ego.trajectory.history_states.keys())
    offset = int(fps * 1.0)
    takeover_idx = min(int(fps * 1.0), len(frames) // 5)
    takeover = frames[takeover_idx]
    hist_n = takeover_idx

    orig_hist = dict(ego.trajectory.history_states)
    orig_frames = sorted(orig_hist.keys())

    def _truncate(at):
        ego.trajectory._history_states = {f: s for f, s in orig_hist.items() if f <= at}
        ego.trajectory._frames = sorted(ego.trajectory._history_states.keys())

    _truncate(takeover)

    # ---- receding-horizon MPC ----
    model = LimSimBehaviorModel(LIMSIM_CFG)
    mpc_xs, mpc_ys, mpc_ts, mpc_plans = [], [], [], {}
    cur = takeover
    dt_ms = int(LIMSIM_CFG.dt * 1000)

    for _ in range(int(num_seconds * 1000 / dt_ms)):
        try:
            pred = model.predict(participants, map_, cur, agent_ids=[ego_id])
        except Exception:
            break
        if ego_id not in pred:
            break
        pt = pred[ego_id]
        ff = [f for f in pt.frames if f > cur]
        if not ff:
            break

        first_f, first_s = ff[0], pt.get_state(ff[0])
        anchor = ego.trajectory.get_state(cur)
        mpc_plans[cur] = [(pt.get_state(f).x - anchor.x, pt.get_state(f).y - anchor.y) for f in ff]
        mpc_xs.append(first_s.x)
        mpc_ys.append(first_s.y)
        mpc_ts.append(first_f)

        if ego.trajectory.has_state(first_f):
            ego.trajectory._history_states[first_f] = first_s
        else:
            ego.trajectory.add_state(first_s)
        cur = first_f

    # ---- interpolate MPC to existing frame numbers ----
    _truncate(takeover)
    target = [f for f in orig_frames if f > takeover]
    StateCls = type(ego.trajectory.get_state(takeover))

    if len(target) > 0 and len(mpc_ts) >= 1:
        if len(mpc_ts) >= 2:
            ix = np.interp(target, mpc_ts, np.array(mpc_xs))
            iy = np.interp(target, mpc_ts, np.array(mpc_ys))
        else:
            ix = np.full(len(target), mpc_xs[0])
            iy = np.full(len(target), mpc_ys[0])
        h = ego.trajectory.get_state(takeover).heading
        for i, (t, x, y) in enumerate(zip(target, ix, iy)):
            if i < len(target) - 1:
                dx, dy = ix[i + 1] - x, iy[i + 1] - y
                if abs(dx) > 1e-6 or abs(dy) > 1e-6:
                    h = np.arctan2(dy, dx)
            ego.trajectory.add_state(StateCls(frame=t, x=x, y=y, heading=float(h)))

    all_frames = frames[takeover_idx - hist_n : takeover_idx] + [
        f for f in target if f > frames[takeover_idx - 1]
    ]
    print(f"  playback: {len(all_frames)} frames  ({all_frames[0]}-{all_frames[-1]})")

    # ---- render ----
    ani = render_takeover_animation(
        participants, map_, all_frames, ego_id, mpc_plans=mpc_plans, resolution=resolution
    )

    return ani

### Example 1: WOMD — 234dfbe99b740c80

This example demonstrates how LimSim takes over selected vehicles in a WOMD scenario by replacing their future trajectories with model predictions. In the animation below, the ego vehicle is shown in pink, its ground-truth trajectory in purple, and the trajectory predicted by LimSim in blue. The takeover is triggered 1 second after the scenario begins, using the ego vehicle's previous 1 second of motion history as input for prediction.

In [7]:
ani_womd = run_takeover_scenario(
    WOMDParser(),
    scenario_id="234dfbe99b740c80",
    file_name=(
        "uncompressed_scenario_validation_interactive_"
        "validation_interactive.tfrecord-00000-of-00150"
    ),
    folder="../../data/WOMD",
    ego_id=364,
)
ani_womd

Parsing scenario ...
  participants: 55,  frames: (0, 8997)
  takeover target: 364  (color: light-pink)
  playback: 90 frames  (0-8997)


### Example 2: HighD (LevelX) — Location 1

HighD uses `LevelXParser("highD")` at 25 Hz on German highway recordings. The map is a lanelet2 `.osm` file parsed with `OSMParser` and `HIGHD_MAP_CONFIG`.  Files 01–06 map to location 1 (`highD_1`). Highway speeds (~30 m/s) produce visibly longer MPC plans than urban datasets like InD.  

Adjust the paths below to match your local data layout.

In [8]:
ani_highd = run_takeover_scenario(
    LevelXParser("highD"),
    file_name=11,
    folder="../../data/highD/data",
    map_path="../../data/highD_map/highD_1.osm",
    map_config=HIGHD_MAP_CONFIG["highD_1"],
    ego_id=18,
    fps=25,
)
ani_highd

Parsing scenario ...


  loading map from ../../data/highD_map/highD_1.osm
  participants: 1776,  frames: (np.int64(40), np.int64(611080))
  takeover target: 18  (color: light-pink)
  playback: 240 frames  (40-9640)


### Example 3: inD (LevelX) — Location 1

This example demonstrates LimSim on the **inD** dataset using `LevelXParser("inD")`, which processes the data at **25 Hz**. The road network is loaded from a Lanelet2 `.osm` map using `OSMParser` together with `IND_MAP_CONFIG`. Recordings **00–06** correspond to **Location 1** (`inD_1`), **07–17** to **Location 2**, and the remaining recordings to the other inD locations.

In this scenario, the ego vehicle approaches an intersection where the intentions of surrounding vehicles are ambiguous. To ensure safe interaction under this uncertainty, LimSim keeps the controlled vehicle stationary until a safe maneuver can be determined.

In [9]:
ani_ind = run_takeover_scenario(
    LevelXParser("inD"),
    file_name=7,
    folder="../../data/inD/data",
    map_path="../../data/inD_map/inD_1.osm",
    map_config=IND_MAP_CONFIG["inD_1"],
    ego_id=12,
    fps=25,
)
ani_ind

Parsing scenario ...
  loading map from ../../data/inD_map/inD_1.osm
  participants: 212,  frames: (np.int64(0), np.int64(1055240))
  takeover target: 12  (color: light-pink)


  playback: 161 frames  (10240-16680)


## Parameter Impact Analysis

The following table summarises how `LimSimConfig` parameters affect runtime and behavior. Adjust these based on your accuracy-vs-speed trade-off.

### Runtime-dominant Parameters

| Parameter | Effect | Guidance |
|-----------|--------|----------|
| `mcts_iterations` | **Primary** runtime driver. Each iteration evaluates candidate actions for every agent in a group. | 15–30 for demos; 80–200 for evaluation. Doubling roughly doubles `plan()` latency. |
| `horizon_steps` | Controls trajectory length and Frenet planner rollout depth. | 10 (1 s) for quick previews; 30–50 (3–5 s) for closed-loop simulation. |
| `max_group_size` | Caps the number of agents in a joint MCTS group. MCTS complexity grows combinatorially with group size. | 3–4 for interactive scenarios; 1 disables joint planning. |
| `interaction_distance` | Determines how many agents are grouped together. Larger values create bigger groups. | 30–50 m for urban; reduce to 15–20 m to shrink groups and speed up planning. |
| `parallel_workers` | Thread pool size for `predict_batch`. Scales nearly linearly up to CPU core count. | 4–8 for laptops; 16+ for workstations. Set to 0 to disable. |

### Behavior-dominant Parameters

| Parameter | Effect | Guidance |
|-----------|--------|----------|
| `collision_penalty` | Penalty weight for predicted collisions in MCTS scoring. Higher values produce more conservative behavior. | 500–2000. Increase if agents drive too aggressively. |
| `progress_weight` | Reward for forward progress along the route. Higher values encourage faster, more goal-directed driving. | 1.0–5.0. Increase for highway, decrease for dense intersections. |
| `speed_weight` | Reward for maintaining target speed. Balances against progress and comfort. | 0.1–0.5. Increase if agents drive too slowly. |
| `lane_change_penalty` | Cost of initiating a lane change. Discourages unnecessary weaving. | 1.0–5.0. Increase for conservative lane-keeping. |
| `candidate_actions` | The set of actions MCTS can choose from. Fewer actions = faster but less flexible. | Default: `(KS, AC, DC, LCL, LCR)`. Drop `LCL`/`LCR` to disable lane changes. |
| `comfort_weight` | Penalty on lateral acceleration and jerk. Higher values produce smoother trajectories. | 0.2–1.0. Increase for passenger-comfort scenarios. |
| `decision_resolution` | Spatial resolution for MCTS action sampling. Finer resolution = more precise but more iterations needed. | 1.0–2.0 m. Match to lane width. |

### Quick Configurations

| Use Case | Key Settings |
|----------|-------------|
| **Fast preview** | `mcts_iterations=10`, `horizon_steps=10`, `max_group_size=2`, `interaction_distance=20` |
| **Balanced demo** (current) | `mcts_iterations=15`, `horizon_steps=10`, `max_group_size=4`, `interaction_distance=30` |
| **High-fidelity eval** | `mcts_iterations=80`, `horizon_steps=30`, `max_group_size=4`, `interaction_distance=50` |

## Performance Analysis

The benchmark follows the original LimSim paper's receding-horizon control approach:
at each timestep, LimSim plans for ``horizon_steps × dt`` (1 s window), executes only the
**first step**, feeds it back into the ego trajectory as an observation, then re-plans.  This
loops for 10 seconds, producing 100 MPC cycles at 10 Hz (250 at 25 Hz).

Each dataset contributes 100 random takeover scenarios, each evaluated under three
configurations (fast preview, balanced demo, high-fidelity eval).  The scatter points are
individual ``model.predict()`` calls; solid curves are quadratic fits to the binned data.

In [ ]:
DATASETS = ["highD", "inD", "rounD", "WOMD"]
CONFIGS = ["fast_preview", "balanced_demo", "high_fidelity"]
CFG_COLORS = dict(zip(CONFIGS, husl))
CFG_MARKERS = {"fast_preview": "o", "balanced_demo": "s", "high_fidelity": "^"}

In [11]:
def plot_benchmark_analysis(df, datasets=None, configs=None, cfg_colors=None):
    datasets = datasets or DATASETS
    configs = configs or CONFIGS
    cfg_colors = cfg_colors or CFG_COLORS
    METRICS = {
        "pred_time_s": ("prediction time (s)", 0),
        "collision_risk": ("collision rate", 0.03),
    }
    fig, axes = plt.subplots(len(datasets), 2, figsize=(12, 3 * len(datasets)))
    rng = np.random.default_rng(42)
    for row, ds in enumerate(datasets):
        sub = df[df["dataset"] == ds]
        for col, (key, (ylabel, jitter)) in enumerate(METRICS.items()):
            ax = axes[row, col]
            for cfg in configs:
                cfg_sub = sub[sub["config"] == cfg]
                x = cfg_sub["avg_active_participants"].values
                y = cfg_sub[key].values
                if len(x) < 3:
                    continue
                yj = y + rng.uniform(-jitter, jitter, len(y)) if jitter else y
                if jitter:
                    yj = yj.clip(-0.02, 1.02)
                ax.scatter(
                    x,
                    yj,
                    c=[cfg_colors[cfg]],
                    alpha=0.2,
                    s=16,
                    marker=CFG_MARKERS[cfg],
                    edgecolors="none",
                )
                idx = np.argsort(x)
                p = np.poly1d(np.polyfit(x[idx], y[idx], 2))
                xl = np.linspace(x.min(), x.max(), 80)
                yl = np.clip(p(xl), 0, 1) if jitter else p(xl)
                ax.plot(xl, yl, c=cfg_colors[cfg], lw=2, label=cfg.replace("_", " "))
            ax.set_xlabel("avg active participants")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{ds} — {ylabel}")
            if row == 0 and col == 0:
                ax.legend(fontsize=7, ncol=3, loc="upper left")
    fig.tight_layout(pad=2)
    return fig

In [ ]:
df = pd.read_csv("../../data/limsim_summary.csv")
plot_benchmark_analysis(df, cfg_colors=CFG_COLORS)

FileNotFoundError: ../../runtime/limsim/summary.csv not found — run benchmark first.